# OptiCrop – Exploratory Data Analysis & Model Comparison

This notebook analyzes the **Crop Recommendation Dataset** and evaluates three classification algorithms:
1. **Random Forest Classifier**
2. **XGBoost Classifier**
3. **Support Vector Machine (SVM)**

The goal is to select the best model to deploy in our FastAPI backend for predicting suitable crops based on soil and weather parameters.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# Print settings
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load the Dataset

In [ ]:
csv_path = "../Crop_recommendation.csv"
if not os.path.exists(csv_path):
    csv_path = "Crop_recommendation.csv"
    
df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

### Crop Label Distribution
Let's see if the crop labels are balanced.

In [ ]:
plt.figure(figsize=(15, 6))
sns.countplot(data=df, x='label', order=df['label'].value_counts().index, palette='viridis')
plt.title("Distribution of Crops in the Dataset", fontsize=16)
plt.xlabel("Crop Type", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Feature Correlation Analysis
Let's check if the soil and weather parameters have strong correlations.

In [ ]:
plt.figure(figsize=(10, 8))
numeric_df = df.drop(columns=['label'])
sns.heatmap(numeric_df.corr(), annot=True, cmap="YlGnBu", fmt=".2f", cbar=True, square=True)
plt.title("Correlation Matrix of Environmental Features", fontsize=16)
plt.tight_layout()
plt.show()

### Feature Distributions Per Crop
Let's analyze feature distribution ranges across crops using box plots.

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for idx, feat in enumerate(features):
    sns.boxplot(data=df, x='label', y=feat, ax=axes[idx], palette='Set2')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=90)
    axes[idx].set_title(f"{feat} Distribution per Crop", fontsize=14)
    axes[idx].set_xlabel("")
    
# Hide unused last plot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
X = df[features]
y = df['label']

# Label Encode Target variable (string to integer)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scaler Initialization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

## 4. Model Training & Evaluation

Let's compare the performance of Random Forest, XGBoost, and Support Vector Machine.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')
    results[name] = {"accuracy": acc, "f1_score": f1, "model": model}
    
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted F1-Score: {f1:.4f}\n")

## 5. Visualizing Confusion Matrix & Feature Importance

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1_score'])
best_model = results[best_name]['model']
preds = best_model.predict(X_test_scaled)

plt.figure(figsize=(15, 12))
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap="Greens")
plt.title(f"Confusion Matrix for Best Performing Model: {best_name}", fontsize=16)
plt.xlabel("Predicted Crop Label")
plt.ylabel("True Crop Label")
plt.tight_layout()
plt.show()

In [ ]:
# Plot feature importance if supported by model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances[indices], y=[features[i] for i in indices], palette='mako')
    plt.title(f"Feature Importance Profile ({best_name})", fontsize=16)
    plt.xlabel("Relative Importance Score")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print(f"The best model ({best_name}) does not support direct feature importance inspection.")